In [ ]:
# 1. 导入必要库（和之前一致，必须用 scipy.sparse）
import os
import scipy.sparse as sp
from pathlib import Path
import re

# 2. 自动定位矩阵文件（支持 N/n 大小写差异、不同启动目录）
target_l, target_n = 6, 3
matrix_name_candidates = [
    f"L={target_l} N={target_n}.npz",
    f"L={target_l} n={target_n}.npz",
    f"l={target_l} N={target_n}.npz",
    f"l={target_l} n={target_n}.npz",
]
cwd = Path.cwd()

def _normalize_filename(name: str) -> str:
    return "".join(name.lower().split())

matrix_file = None
matrix_override = os.environ.get("MATRIX_FILE")
if matrix_override:
    override_path = Path(matrix_override).expanduser()
    if override_path.exists():
        matrix_file = override_path
    else:
        print(f"⚠️ MATRIX_FILE 不存在: {override_path}")

candidate_roots = [cwd, cwd / "lunwen", cwd.parent, cwd.parent / "lunwen"]
matrix_candidates = [root / name for root in candidate_roots for name in matrix_name_candidates]
if matrix_file is None:
    matrix_file = next((p for p in matrix_candidates if p.exists()), None)

search_roots = [cwd, cwd.parent, cwd.parent.parent]
if matrix_file is None:
    for root in search_roots:
        if not root.exists():
            continue
        for name in matrix_name_candidates:
            matches = sorted(root.rglob(name))
            if matches:
                matrix_file = matches[0]
                break
        if matrix_file is not None:
            break

if matrix_file is None:
    expected_norms = {_normalize_filename(name) for name in matrix_name_candidates}
    fuzzy_hits = []
    npz_inventory = []
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("*.npz"):
            npz_inventory.append(p)
            name_norm = _normalize_filename(p.name)
            if name_norm in expected_norms:
                fuzzy_hits.append(p)
                continue
            has_l = re.search(rf"l\\s*=\\s*{target_l}\\b", p.name, flags=re.IGNORECASE)
            has_n = re.search(rf"n\\s*=\\s*{target_n}\\b", p.name, flags=re.IGNORECASE)
            if has_l and has_n:
                fuzzy_hits.append(p)

    if fuzzy_hits:
        matrix_file = sorted(set(fuzzy_hits), key=lambda x: (len(str(x)), str(x)))[0]
    else:
        tried = [str(p) for p in matrix_candidates]
        sample = [str(p) for p in sorted(set(npz_inventory), key=lambda x: str(x))[:20]]
        scanned = [str(p) for p in search_roots]
        raise FileNotFoundError("Cannot find matrix file for L=6,N=3. cwd={cwd}, tried={tried}, scanned={scanned}, npz_sample={sample}")

H_3 = sp.load_npz(str(matrix_file))
print(f"使用矩阵文件: {matrix_file}")
